<a href="https://colab.research.google.com/github/alishadaudi/data-science-project-dfw-red-cross-donor-prediction/blob/main/DFW_Red_Cross_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Code snippet 1
# Installing pycaret for ML modeling

!pip install -U pip setuptools wheel

!pip install -U pycaret

In [ ]:
# Code snippet 2
# Importing classification components from pycaret
from pycaret.classification import *

In [ ]:
# Code snippet 3
# Importing the DFW Red Cross dataset
import pandas as pd
data = pd.read_csv ('https://docs.google.com/spreadsheets/d/e/2PACX-1vQzBYWDif8AqH47QpdaMsxZ0d3aXafgvL6EfnsUk6iN5QPCgrhvEky7hzI16iyfL3L2rfec3QX32JQj/pub?gid=0&single=true&output=csv')

# Drop 'CumulativeDonationAmount' due to data leakage
# This column directly reflects the target and should not be used as a predictor.
if 'CumulativeDonationAmount' in data.columns:
    data = data.drop(columns=['CumulativeDonationAmount'])

# List of columns that contain monetary values as strings
monetary_cols_to_clean = [
    'LastFiscalYearDonation', 'Donation2FiscalYearsAgo',
    'Donation3FiscalYearsAgo', 'Donation4FiscalYearsAgo',
    'Donation5FiscalYearsAgo', 'CurrentFiscalYearDonation'
]

# Clean and convert monetary columns to numeric immediately after data loading
for col in monetary_cols_to_clean:
    if col in data.columns:
        data[col] = data[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# Display the data
data

,DonorUniqueId,DonorPostalCode,DonorAge,MaritalStatus,GenderIdentity,IsMemberFlag,IsAlumnusFlag,IsParentFlag,HasInvolvementFlag,WealthRating,...,HasEmailFlag,ConsecutiveDonorYears,LastFiscalYearDonation,Donation2FiscalYearsAgo,Donation3FiscalYearsAgo,Donation4FiscalYearsAgo,Donation5FiscalYearsAgo,CurrentFiscalYearDonation,DonorDateOfBirth,DonorIndicatorFlag.
0,1,23187.0,42,Married,Female,N,N,N,N,NaN,...,N,1,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Y
1,2,77643.0,33,NaN,Female,N,Y,N,Y,NaN,...,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,6/16/1985,Y
2,3,NaN,42,Married,Female,N,N,N,N,NaN,...,N,1,0.0,0.0,0.0,0.0,0.0,200.0,NaN,Y
3,4,47141.0,31,NaN,Female,N,Y,N,Y,NaN,...,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,12/3/1987,N
4,5,92555.0,68,NaN,Female,N,N,N,N,NaN,...,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,9/11/1950,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34503,34504,7848.0,42,NaN,Female,N,N,N,N,NaN,...,N,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,N
34504,34505,28275.0,24,NaN,Male,N,N,N,N,"$250,000-$499,999",...,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,9/23/1994,Y
34505,34506,42539.0,27,NaN,Female,N,Y,N,Y,NaN,...,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,1/3/1991,N
34506,34507,32733.0,46,Married,Female,N,N,N,Y,NaN,...,Y,1,0.0,0.0,0.0,120.0,0.0,0.0,5/11/1972,Y


In [ ]:
# Code snippet 4
# Displaying the dataset columns
data.columns

Index(['DonorPostalCode', 'DonorAge', 'MaritalStatus', 'GenderIdentity',
       'IsMemberFlag', 'IsAlumnusFlag', 'IsParentFlag', 'HasInvolvementFlag',
       'PreferredAddressType', 'HasEmailFlag', 'ConsecutiveDonorYears',
       'LastFiscalYearDonation', 'Donation2FiscalYearsAgo',
       'Donation3FiscalYearsAgo', 'Donation4FiscalYearsAgo',
       'Donation5FiscalYearsAgo', 'CurrentFiscalYearDonation',
       'DonorIndicatorFlag.'],
      dtype='object')

In [ ]:
# Code snippet 5
# Displaying missing data percentage for every column
data.isnull().sum()/len(data)*100

,0
DonorPostalCode,0.263707
DonorAge,0.000000
MaritalStatus,71.195085
GenderIdentity,1.428654
IsMemberFlag,0.000000
IsAlumnusFlag,0.000000
IsParentFlag,0.000000
HasInvolvementFlag,0.000000
PreferredAddressType,11.716124
HasEmailFlag,0.000000


In [ ]:
# Code snippet 6
# Setting the Target variable
y = 'DonorIndicatorFlag.'

In [ ]:
# Code snippet 7
# Setting the numerical variables
num_cols = [
    'DonorAge',
    'ConsecutiveDonorYears',
    'LastFiscalYearDonation',
    'Donation2FiscalYearsAgo',
    'Donation3FiscalYearsAgo',
    'Donation4FiscalYearsAgo',
    'Donation5FiscalYearsAgo',
    'CurrentFiscalYearDonation'
]

In [ ]:
# Code snippet 8
# Setting the categorical columns
cat_cols = [
    'DonorPostalCode',
    'MaritalStatus',
    'GenderIdentity',
    'IsMemberFlag',
    'IsAlumnusFlag',
    'IsParentFlag',
    'HasInvolvementFlag',
    'PreferredAddressType',
    'HasEmailFlag',
    'AcademicDegreeLevel', # Add original column for PyCaret to handle as ordinal
    'WealthRating' # Add original column for PyCaret to handle as ordinal
]

In [ ]:
# Code snippet 9
# Drop Identifier/Redundant Columns
# These columns were likely dropped by pycaret.setup already.
# data = data.drop(columns=['DonorUniqueId', 'DonorDateOfBirth'])

In [ ]:
# Code snippet 10
# Extracting the first 3 digits from 'DonorPostalCode' to reduce cardinality
# Convert to string first to handle potential float/NaN values, then extract or fill NaN
data['DonorPostalCode'] = data['DonorPostalCode'].astype(str).str.slice(0, 3).replace('nan', pd.NA)

In [ ]:
# Code snippet 11
import pandas as pd
from pycaret.classification import setup

# Reload the original dataset to ensure a clean state before PyCaret setup
data = pd.read_csv ('https://docs.google.com/spreadsheets/d/e/2PACX-1vQzBYWDif8AqH47QpdaMsxZ0d3aXafgvL6EfnsUk6iN5QPCgrhvEky7hzI16iyfL3L2rfec3QX32JQj/pub?gid=0&single=true&output=csv')

# Define the target variable again after reloading data
y = 'DonorIndicatorFlag.'

# 1. Drop 'CumulativeDonationAmount' due to data leakage
# This column directly reflects the target and should not be used as a predictor.
if 'CumulativeDonationAmount' in data.columns:
    data = data.drop(columns=['CumulativeDonationAmount'])

# 2. List of columns that contain monetary values as strings
monetary_cols_to_clean = [
    'LastFiscalYearDonation', 'Donation2FiscalYearsAgo',
    'Donation3FiscalYearsAgo', 'Donation4FiscalYearsAgo',
    'Donation5FiscalYearsAgo', 'CurrentFiscalYearDonation'
]

# Clean and convert monetary columns to numeric
for col in monetary_cols_to_clean:
    if col in data.columns:
        data[col] = data[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# 3. Extract the first 3 digits from 'DonorPostalCode' to reduce cardinality
# Convert to string first to handle potential float/NaN values, then extract or fill NaN
data['DonorPostalCode'] = data['DonorPostalCode'].astype(str).str.slice(0, 3).replace('nan', pd.NA)

# 4. Define columns to drop before setup, including those with high missingness (>20% rule)
columns_to_drop_before_setup = ['DonorUniqueId', 'DonorDateOfBirth', 'AcademicDegreeLevel', 'WealthRating']
data = data.drop(columns=[col for col in columns_to_drop_before_setup if col in data.columns])

# Update categorical columns after dropping 'AcademicDegreeLevel' and 'WealthRating'
# These are the original categorical features PyCaret will one-hot encode.
updated_cat_cols = [
    'DonorPostalCode',
    'MaritalStatus',
    'GenderIdentity',
    'IsMemberFlag',
    'IsAlumnusFlag',
    'IsParentFlag',
    'HasInvolvementFlag',
    'PreferredAddressType',
    'HasEmailFlag'
]

# Re-initialize num_cols to its original numerical features
# PyCaret will handle the scaling of these when `normalize=True`
updated_num_cols = [
    'DonorAge',
    'ConsecutiveDonorYears',
    'LastFiscalYearDonation',
    'Donation2FiscalYearsAgo',
    'Donation3FiscalYearsAgo',
    'Donation4FiscalYearsAgo',
    'Donation5FiscalYearsAgo',
    'CurrentFiscalYearDonation'
]

classification_setup = setup(data,
                            target = y,
                            numeric_features = updated_num_cols,
                            categorical_features = updated_cat_cols,
                            normalize = True, # Enable numerical feature scaling
                            fix_imbalance = True, # Added to address class imbalance
                            session_id=8502) # For reproducibility


,Description,Value
0,Session id,8502
1,Target,DonorIndicatorFlag.
2,Target type,Binary
3,Target mapping,"N: 0, Y: 1"
4,Original data shape,"(34508, 18)"
5,Transformed data shape,"(40359, 30)"
6,Transformed train set shape,"(30006, 30)"
7,Transformed test set shape,"(10353, 30)"
8,Numeric features,8
9,Categorical features,9


In [ ]:
# Code snippet 12
# Invoking ML algorithms
compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ada,Ada Boost Classifier,0.6877,0.7549,0.6877,0.7907,0.6856,0.4167,0.4810,1.6740
gbc,Gradient Boosting Classifier,0.6872,0.7557,0.6872,0.7843,0.6858,0.4136,0.4733,3.2790
lightgbm,Light Gradient Boosting Machine,0.6855,0.7542,0.6855,0.7682,0.6858,0.4047,0.4532,2.2570
qda,Quadratic Discriminant Analysis,0.6837,0.7572,0.6837,0.8244,0.6766,0.4213,0.5138,1.0500
xgboost,Extreme Gradient Boosting,0.6835,0.7541,0.6835,0.7540,0.6852,0.3951,0.4343,1.3730
dt,Decision Tree Classifier,0.6683,0.6744,0.6683,0.6814,0.6721,0.3174,0.3203,1.1080
rf,Random Forest Classifier,0.6677,0.7476,0.6677,0.6839,0.6720,0.3204,0.3243,4.2180
et,Extra Trees Classifier,0.6560,0.6873,0.6560,0.6739,0.6606,0.2986,0.3029,5.8590
nb,Naive Bayes,0.6556,0.7489,0.6556,0.8164,0.6437,0.3777,0.4799,0.9950
lr,Logistic Regression,0.5705,0.6281,0.5705,0.6180,0.5753,0.1650,0.1763,2.1340


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

AdaBoostClassifier(algorithm='SAMME.R', estimator=None, learning_rate=1.0,
                   n_estimators=50, random_state=8502)

In [ ]:
# Code snippet 13
# Cross validation/building the best model with the best algo
best_model = create_model('ada')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7065,0.7762,0.7065,0.8136,0.7044,0.4522,0.5227
1,0.6929,0.7613,0.6929,0.7967,0.6908,0.4264,0.4922
2,0.6983,0.7630,0.6983,0.7853,0.6983,0.4299,0.4835
3,0.6734,0.7439,0.6734,0.7940,0.6685,0.3972,0.4725
4,0.6838,0.7518,0.6838,0.7910,0.6810,0.4112,0.4781
5,0.6894,0.7478,0.6894,0.7885,0.6879,0.4185,0.4801
6,0.6845,0.7513,0.6845,0.7760,0.6837,0.4065,0.4615
7,0.6774,0.7466,0.6774,0.7832,0.6747,0.3993,0.4641
8,0.6816,0.7592,0.6816,0.7786,0.6801,0.4034,0.4622


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Code snippet 14
# Evaluating the model (to get confusion matrix and feature importance)
evaluate_model(best_model)

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [ ]:
# Code snippet 15
# Importing the client dataset
client_data = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vQzBYWDif8AqH47QpdaMsxZ0d3aXafgvL6EfnsUk6iN5QPCgrhvEky7hzI16iyfL3L2rfec3QX32JQj/pub?gid=0&single=true&output=csv')

# Displaying client data
client_data

,DonorUniqueId,DonorPostalCode,DonorAge,MaritalStatus,GenderIdentity,IsMemberFlag,IsAlumnusFlag,IsParentFlag,HasInvolvementFlag,WealthRating,...,ConsecutiveDonorYears,LastFiscalYearDonation,Donation2FiscalYearsAgo,Donation3FiscalYearsAgo,Donation4FiscalYearsAgo,Donation5FiscalYearsAgo,CurrentFiscalYearDonation,CumulativeDonationAmount,DonorDateOfBirth,DonorIndicatorFlag.
0,1,23187.0,42,Married,Female,N,N,N,N,NaN,...,1,$0,$0,$0,$0,$0,$0,10.0,NaN,Y
1,2,77643.0,33,NaN,Female,N,Y,N,Y,NaN,...,0,$0,$0,$0,$0,$0,$0,2100.0,6/16/1985,Y
2,3,NaN,42,Married,Female,N,N,N,N,NaN,...,1,$0,$0,$0,$0,$0,$200,200.0,NaN,Y
3,4,47141.0,31,NaN,Female,N,Y,N,Y,NaN,...,0,$0,$0,$0,$0,$0,$0,0.0,12/3/1987,N
4,5,92555.0,68,NaN,Female,N,N,N,N,NaN,...,0,$0,$0,$0,$0,$0,$0,505.0,9/11/1950,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34503,34504,7848.0,42,NaN,Female,N,N,N,N,NaN,...,0,$0,$0,$0,$0,$0,$0,0.0,NaN,N
34504,34505,28275.0,24,NaN,Male,N,N,N,N,"$250,000-$499,999",...,0,$0,$0,$0,$0,$0,$0,80.0,9/23/1994,Y
34505,34506,42539.0,27,NaN,Female,N,Y,N,Y,NaN,...,0,$0,$0,$0,$0,$0,$0,0.0,1/3/1991,N
34506,34507,32733.0,46,Married,Female,N,N,N,Y,NaN,...,1,$0,$0,$0,$120,$0,$0,120.0,5/11/1972,Y


In [ ]:
# Code snippet 16
# Predicting the donor indicator flag (DonorIndicatorFlag.) for the client data set

# Re-load client data as well to ensure consistent preprocessing
client_data = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vQzBYWDif8AqH47QpdaMsxZ0d3aXafgvL6EfnsUk6iN5QPCgrhvEky7hzI16iyfL3L2rfec3QX32JQj/pub?gid=0&single=true&output=csv')

# 1. Drop 'CumulativeDonationAmount' due to data leakage
# This column directly reflects the target and should not be used as a predictor.
if 'CumulativeDonationAmount' in client_data.columns:
    client_data = client_data.drop(columns=['CumulativeDonationAmount'])

# 2. Clean and convert monetary columns to numeric
monetary_cols_to_clean = [
    'LastFiscalYearDonation', 'Donation2FiscalYearsAgo',
    'Donation3FiscalYearsAgo', 'Donation4FiscalYearsAgo',
    'Donation5FiscalYearsAgo', 'CurrentFiscalYearDonation'
]

for col in monetary_cols_to_clean:
    if col in client_data.columns:
        client_data[col] = client_data[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# 3. Extract the first 3 digits from 'DonorPostalCode'
client_data['DonorPostalCode'] = client_data['DonorPostalCode'].astype(str).str.slice(0, 3).replace('nan', pd.NA)

# 4. Drop columns that were dropped from training data before setup
columns_to_drop_before_predict = ['DonorUniqueId', 'DonorDateOfBirth', 'AcademicDegreeLevel', 'WealthRating']
client_data = client_data.drop(columns=[col for col in columns_to_drop_before_predict if col in client_data.columns])

predictions = predict_model(best_model, client_data)

# Exporting to a CSV file
predictions.to_csv('DFW_RedCross_Data_Predictions_.csv')

# Display the predictions
predictions

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Ada Boost Classifier,0.6999,0.7870,0.6999,0.8092,0.6973,0.4409,0.5120


,DonorPostalCode,DonorAge,MaritalStatus,GenderIdentity,IsMemberFlag,IsAlumnusFlag,IsParentFlag,HasInvolvementFlag,PreferredAddressType,HasEmailFlag,ConsecutiveDonorYears,LastFiscalYearDonation,Donation2FiscalYearsAgo,Donation3FiscalYearsAgo,Donation4FiscalYearsAgo,Donation5FiscalYearsAgo,CurrentFiscalYearDonation,DonorIndicatorFlag.,prediction_label,prediction_score
0,231,42,Married,Female,N,N,N,N,HOME,N,1,0.0,0.0,0.0,0.0,0.0,0.0,Y,N,0.5045
1,776,33,NaN,Female,N,Y,N,Y,NaN,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,Y,N,0.5038
2,NaN,42,Married,Female,N,N,N,N,HOME,N,1,0.0,0.0,0.0,0.0,0.0,200.0,Y,Y,0.6701
3,471,31,NaN,Female,N,Y,N,Y,HOME,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,N,N,0.5038
4,925,68,NaN,Female,N,N,N,N,HOME,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,Y,N,0.5016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34503,784,42,NaN,Female,N,N,N,N,HOME,N,0,0.0,0.0,0.0,0.0,0.0,0.0,N,N,0.5013
34504,282,24,NaN,Male,N,N,N,N,CAMP,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,Y,Y,0.5032
34505,425,27,NaN,Female,N,Y,N,Y,HOME,Y,0,0.0,0.0,0.0,0.0,0.0,0.0,N,N,0.5038
34506,327,46,Married,Female,N,N,N,Y,HOME,Y,1,0.0,0.0,0.0,120.0,0.0,0.0,Y,Y,0.6719
